# LightFM Hybrid

Hybrid recommender: interaction + item metadata. CPU. Checkpoint ghi đè `latest.joblib` mỗi epoch.

In [ ]:

from pathlib import Path
import sys

PROJECT_ROOT = Path(r"D:\MerRec")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)


## Chuẩn bị sparse interaction + metadata

In [ ]:

import json, os
import numpy as np
import scipy.sparse as sp
import duckdb, joblib
import pyarrow.parquet as pq

from training.utils.recommender_data import (
    prepare_active_universe, CF_ACTIVE, ITEM_FEATURES, load_shared_stats
)
from training.utils.common import stable_hash
from training.utils.paths import CHECKPOINT_DIR

prepare_active_universe(target_interaction_coverage=0.95)
stats = load_shared_stats()

con = duckdb.connect()
a = con.execute(f"""
    SELECT
        user_idx::BIGINT user_idx,
        item_idx::BIGINT item_idx,
        LN(1 + implicit_score)::FLOAT w
    FROM read_parquet('{CF_ACTIVE.as_posix()}')
""").fetchnumpy()
con.close()

u = a["user_idx"].astype(np.int64, copy=False)
i = a["item_idx"].astype(np.int64, copy=False)
w = a["w"].astype(np.float32, copy=False)

shape = (int(stats["users"]), int(stats["items"]))
interactions = sp.coo_matrix(
    (np.ones_like(w, dtype=np.float32),(u,i)),
    shape=shape
).tocsr()

sample_weight = sp.coo_matrix(
    (w,(u,i)),
    shape=shape
).tocsr()

print(shape, interactions.nnz)


## Item features

In [ ]:

LIGHTFM_HASH_BUCKETS = 65_536
n_items = int(stats["items"])

df = pq.read_table(
    ITEM_FEATURES,
    columns=["item_idx","category0","category1","category2","brand","condition"]
).to_pandas()

rows, cols, vals = [], [], []

for r in df.itertuples(index=False):
    idx = int(r.item_idx)

    # item identity
    rows.append(idx)
    cols.append(idx)
    vals.append(1.0)

    for value, prefix in [
        (r.category0,"c0"),
        (r.category1,"c1"),
        (r.category2,"c2"),
        (r.brand,"brand"),
        (r.condition,"condition"),
    ]:
        h = stable_hash(
            f"{prefix}={value}",
            LIGHTFM_HASH_BUCKETS,
            seed=42
        )
        rows.append(idx)
        cols.append(n_items+h)
        vals.append(1.0)

item_features = sp.coo_matrix(
    (
        np.asarray(vals,np.float32),
        (
            np.asarray(rows,np.int64),
            np.asarray(cols,np.int64)
        )
    ),
    shape=(n_items, n_items+LIGHTFM_HASH_BUCKETS),
    dtype=np.float32
).tocsr()

print(item_features.shape, item_features.nnz)


## Train + resume

In [ ]:

from lightfm import LightFM

LIGHTFM_EPOCHS = 15

OUT = CHECKPOINT_DIR / "lightfm"
OUT.mkdir(parents=True, exist_ok=True)

LATEST = OUT / "latest.joblib"
STATE = OUT / "state.json"

if LATEST.exists():
    print("🔄 Resume:", LATEST)
    model = joblib.load(LATEST)
    start_epoch = (
        json.loads(STATE.read_text(encoding="utf-8"))["epoch"] + 1
        if STATE.exists() else 0
    )
else:
    model = LightFM(
        no_components=32,
        loss="warp",
        learning_rate=0.05,
        item_alpha=1e-6,
        user_alpha=1e-6,
        random_state=42
    )
    start_epoch = 0

for epoch in range(start_epoch, LIGHTFM_EPOCHS):
    print(f"\nLightFM epoch {epoch+1}/{LIGHTFM_EPOCHS}")
    model.fit_partial(
        interactions,
        item_features=item_features,
        sample_weight=sample_weight,
        epochs=1,
        num_threads=max(1,(os.cpu_count() or 4)-2),
        verbose=True
    )

    tmp = OUT / "latest_tmp.joblib"
    joblib.dump(model, tmp, compress=0)
    tmp.replace(LATEST)

    STATE.write_text(
        json.dumps({"epoch": epoch}, indent=2),
        encoding="utf-8"
    )

    print("✅ saved:", LATEST)
